In [ ]:
# 1. NoLA workspace and verified download helper — no training.
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, os, shutil, subprocess, sys, zipfile
import torch

ROOT = Path('/content/nola_research')
DATA, WEIGHTS, SOURCE = (ROOT / p for p in ('data', 'weights', 'source'))
for p in (DATA, WEIGHTS, SOURCE): p.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Runtime:', torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'CPU', '| PyTorch:', torch.__version__)

def digest(path, kind='sha256'):
    with path.open('rb') as f: return hashlib.file_digest(f, kind).hexdigest()

def fetch(url, dest, expected, kind='sha256'):
    if dest.is_file() and digest(dest, kind) == expected:
        return dest
    tmp = dest.with_suffix(dest.suffix + '.part')
    tmp.unlink(missing_ok=True)
    p = subprocess.run(['curl','--fail','--location','--silent','--show-error','--retry','2','--connect-timeout','15','--max-time','150','--output',str(tmp),url],capture_output=True,text=True,timeout=170)
    if p.returncode: raise RuntimeError(f'{dest.name}: {p.stderr}')
    if digest(tmp, kind) != expected:
        tmp.unlink(missing_ok=True)
        raise ValueError(f'Checksum mismatch: {dest.name}')
    tmp.replace(dest)
    return dest
print('Workspace:', ROOT)

In [ ]:
# 2. Download the full EuroSAT RGB archive, then verify the published CoOp split.
from collections import Counter
from pathlib import PurePosixPath

EUROSAT_MD5 = 'f46e308c4d50d4bf32fedad2d3d62f3b'
SPLIT_SHA256 = '541503a77d0b26b735d85ec7a3e672804fe0aa466c2e5f07da772cc6c0af8ad2'
archive = fetch('https://zenodo.org/api/records/7711810/files/EuroSAT_RGB.zip/content', DATA/'EuroSAT_RGB.zip', EUROSAT_MD5, 'md5')
split_file = DATA/'split_zhou_EuroSAT.json'
if not split_file.is_file() or digest(split_file) != SPLIT_SHA256:
    import importlib.util
    if importlib.util.find_spec('gdown') is None:
        subprocess.run([sys.executable,'-m','pip','-q','install','gdown'],check=True,timeout=60)
    import gdown
    temp = split_file.with_suffix('.part')
    gdown.download(id='1Ip7yaCWFi0eaOFUGga0lUdVi_DDQth1o',output=str(temp),quiet=True)
    assert temp.is_file() and digest(temp) == SPLIT_SHA256, 'Published split hash mismatch'
    temp.replace(split_file)
SPLITS = json.loads(split_file.read_text())
expected = {'train': 13500, 'val': 5400, 'test': 8100}
assert {k:len(v) for k,v in SPLITS.items()} == expected
paths = [x[0] for rows in SPLITS.values() for x in rows]
assert len(paths) == len(set(paths)) == 27000
class_map = {int(label):name for rows in SPLITS.values() for _,label,name in rows}
assert set(class_map) == set(range(10)) and all(class_map[int(label)] == name for rows in SPLITS.values() for _,label,name in rows)
CLASS_NAMES = [class_map[i] for i in range(10)]
with zipfile.ZipFile(archive) as z:
    members = [n for n in z.namelist() if n.lower().endswith('.jpg')]
    mapping = {'/'.join(PurePosixPath(n).parts[-2:]): n for n in members}
    assert len(mapping) == len(members) == 27000 and set(mapping) == set(paths)
    IMAGE_DIR = DATA/'images'
    for rel, member in mapping.items():
        relpath = PurePosixPath(rel)
        assert len(relpath.parts) == 2 and not relpath.is_absolute() and '..' not in relpath.parts
        dest = IMAGE_DIR/rel
        if not dest.is_file() or dest.stat().st_size != z.getinfo(member).file_size:
            dest.parent.mkdir(parents=True,exist_ok=True)
            dest.write_bytes(z.read(member))
assert {str(p.relative_to(IMAGE_DIR)) for p in IMAGE_DIR.glob('*/*.jpg')} == set(paths)
print('EuroSAT:',expected,'| 27,000 unique images across',len(CLASS_NAMES),'classes')
print('Images:',IMAGE_DIR,'| Split:',split_file)

In [ ]:
# 3. Download and verify the two frozen backbone checkpoints.
CLIP_SHA256 = '40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af'
DINO_SHA256 = 'bf34ad0f424b9029b593e8dc3ed553bf26e88bcba0d32bf3e62a6209cb64c85e'
weights = [
    (f'https://openaipublic.azureedge.net/clip/models/{CLIP_SHA256}/ViT-B-32.pt', WEIGHTS/'ViT-B-32.pt', CLIP_SHA256),
    ('https://dl.fbaipublicfiles.com/dino/dino_vitbase16_pretrain/dino_vitbase16_pretrain.pth', WEIGHTS/'dino_vitbase16_pretrain.pth', DINO_SHA256),
]
with ThreadPoolExecutor(max_workers=2) as pool:
    CLIP_WEIGHTS, DINO_WEIGHTS = list(pool.map(lambda args: fetch(*args), weights))
for p in (CLIP_WEIGHTS,DINO_WEIGHTS): print(p.name, f'{p.stat().st_size:,} bytes', 'SHA-256 verified')

In [ ]:
# 4. Smoke-test checkpoint loading and one real image on both backbones.
from PIL import Image
from torchvision import transforms as T

DINO_REV = '7c446df5b9f45747937fb0d72314eb9f7b66930a'
DINO_SOURCE = SOURCE/'dino'
DINO_SOURCE.mkdir(exist_ok=True)
for name in ('vision_transformer.py','utils.py'):
    p = DINO_SOURCE/name
    if not p.is_file():
        base = f'https://raw.githubusercontent.com/facebookresearch/dino/{DINO_REV}/'
        subprocess.run(['curl','-fsSL','--max-time','30','-o',str(p.with_suffix('.part')),base+name],check=True,timeout=35)
        p.with_suffix('.part').replace(p)
sys.path.insert(0,str(DINO_SOURCE))
from vision_transformer import vit_base

def image_tensor(path, mean, std):
    tx = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(), T.Normalize(mean,std)])
    with Image.open(path) as img:
        return tx(img.convert('RGB')).unsqueeze(0).to(DEVICE)

sample = IMAGE_DIR/SPLITS['train'][0][0]
# Load CLIP's official TorchScript checkpoint on its native device, then check the visual encoder.
clip_jit = torch.jit.load(str(CLIP_WEIGHTS),map_location=DEVICE).eval()
dino_model = vit_base(patch_size=16,num_classes=0)
dino_model.load_state_dict(torch.load(DINO_WEIGHTS,map_location='cpu',weights_only=True),strict=True)
dino_model = dino_model.eval().requires_grad_(False).to(DEVICE)
with torch.inference_mode():
    d = dino_model(image_tensor(sample,(.485,.456,.406),(.229,.224,.225)))
    if DEVICE.type == 'cuda':
        c = clip_jit.encode_image(image_tensor(sample,(.48145466,.4578275,.40821073),(.26862954,.26130258,.27577711)))
        assert c.shape == (1,512) and torch.isfinite(c).all()
        print('CLIP forward:',tuple(c.shape),c.device)
    else:
        print('CLIP checkpoint loaded; TorchScript forward requires GPU here.')
    assert d.shape == (1,768) and torch.isfinite(d).all()
    print('DINO forward:',tuple(d.shape),d.device)
print('Setup complete. No hypotheses run; no test predictions computed.')